### Transformer Demo (1 sample pair, 1 update) , (N=1, d_model = 4, h = 2, d_ff = 8)

In [1]:
import math
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import torch.nn.functional as F

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available else "cpu"
print(f"Using {device} device")



# device = torch.device(
#     "cuda" if torch.cuda.is_available()
#     else "cpu"
# )


# print(f"Using {device} device")




/Users/leeminhyeok/Desktop/whisper/whisper1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using mps device


In [2]:
d_model = 4
h = 2
N = 1
d_ff = 8
dropout_p = 0.1
max_length = 128

# tokenizer  = AutoTokenizer.from_pretrained(
#     "google-bert/bert-base-multilingual-cased"
# )

# tokenizer.add_special_tokens({
#     "bos_token" : "<BOS>",
#     "eos_token" : "<EOS>"
# })

# vocab_size = 16
# # vocab_size = len(tokenizer)


In [3]:
vocab = [
    "<PAD>",  # 0
    "<BOS>",  # 1
    "<EOS>",  # 2
    "I",      # 3
    "'",      # 4
    "m",      # 5
    "the",    # 6
    "best",   # 7
    "boxer",  # 8
    ".",      # 9
    "나는",    # 10
    "최고",    # 11
    "##의",    # 12
    "복",      # 13
    "##서",    # 14
    "##다",    # 15
]




source_text = "I'm the best boxer."
target_text = "나는 최고의 복서다."



# source_token_ids = tokenizer.encode(
#     source_text,
#     add_special_tokens = False
# )

# target_token_ids = tokenizer.encode(
#     target_text,
#     add_special_tokens = False
# )

# source_input_ids = [
#     *source_token_ids,
#     tokenizer.eos_token_id
# ]

# decoder_input_ids =[
#     tokenizer.bos_token_id,
#     *target_token_ids
# ]



# labels = [
#     *target_token_ids,
#     tokenizer.eos_token_id
# ]


source_tokens = [
    "I", "'", "m", "the", "best", "boxer", "."
]

target_tokens = [
    "나는", "최고", "##의", "복", "##서", "##다", "."
]

token_to_id = {
    token: token_id
    for token_id, token in enumerate(vocab)
}

id_to_token = {
    token_id: token
    for token, token_id in token_to_id.items()
}




vocab_size = len(vocab)  # 16

pad_token_id = token_to_id["<PAD>"]  # 0
bos_token_id = token_to_id["<BOS>"]  # 1
eos_token_id = token_to_id["<EOS>"]  # 2

source_token_ids = [
    token_to_id[token]
    for token in source_tokens
]

target_token_ids = [
    token_to_id[token]
    for token in target_tokens
]

source_input_ids = [
    *source_token_ids,
    eos_token_id,
]

# Decoder 입력: <BOS> + 번역문
decoder_input_ids = [
    bos_token_id,
    *target_token_ids,
]


source_input_ids = torch.tensor(
    [source_input_ids],
    dtype = torch.long,
    device = device
)

decoder_input_ids = torch.tensor(
    [decoder_input_ids],
    dtype = torch.long,
    device = device
)

# labels = torch.tensor(
#     [labels],
#     dtype = torch.long,
#     device = device
# )


labels = torch.tensor(        
    [[10, 11, 12, 13, 14, 15, 9, 2]],     
    dtype=torch.long,
    device=device,
)


print("Vocab size:", vocab_size)

print("Source text:", source_text)
print("Target text:", target_text)

print("Encoder IDs:", source_input_ids.tolist())
print(
    "Encoder tokens:",
    [id_to_token[token_id]
     for token_id in source_input_ids[0].tolist()]
)

print("Decoder input IDs:", decoder_input_ids.tolist())
print(
    "Decoder input tokens:",
    [id_to_token[token_id]
     for token_id in decoder_input_ids[0].tolist()]
)

print("Label IDs:", labels.tolist())
print(
    "Label tokens:",
    [id_to_token[token_id]
     for token_id in labels[0].tolist()]
)


Vocab size: 16
Source text: I'm the best boxer.
Target text: 나는 최고의 복서다.
Encoder IDs: [[3, 4, 5, 6, 7, 8, 9, 2]]
Encoder tokens: ['I', "'", 'm', 'the', 'best', 'boxer', '.', '<EOS>']
Decoder input IDs: [[1, 10, 11, 12, 13, 14, 15, 9]]
Decoder input tokens: ['<BOS>', '나는', '최고', '##의', '복', '##서', '##다', '.']
Label IDs: [[10, 11, 12, 13, 14, 15, 9, 2]]
Label tokens: ['나는', '최고', '##의', '복', '##서', '##다', '.', '<EOS>']


In [4]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model,
        max_length=128,
        dropout_p=0.1
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout_p)

        position = torch.arange(
            max_length,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            1,
            max_length,
            d_model
        )

        pe[0, :, 0::2] = torch.sin(
            position * div_term
        )

        odd_size = pe[0, :, 1::2].shape[-1]

        pe[0, :, 1::2] = torch.cos(
            position * div_term[:odd_size]
        )

        self.register_buffer("pe", pe)

    def forward(self, x):
        token_len = x.size(1)

        pe = self.pe[:, :token_len].to(
            dtype=x.dtype
        )

        return self.dropout(x + pe)

In [5]:
import math
import torch
import torch.nn as nn


class TranslationTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        pad_token_id,
        d_model=10,
        h=2,
        N=1,
        d_ff=40,
        dropout_p=0.1,
        max_length=128
    ):
        super().__init__()


        self.d_model = d_model
        self.pad_token_id = pad_token_id

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_token_id
        )

      
        self.positional_encoding = (
            SinusoidalPositionalEncoding(
                d_model=d_model,
                max_length=max_length,
                dropout_p=dropout_p
            )
        )

        

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=h,
            dim_feedforward=d_ff,
            dropout=dropout_p,
            activation="relu",
            batch_first=True,
            norm_first=False
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=N
        )

  
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=h,
            dim_feedforward=d_ff,
            dropout=dropout_p,
            activation="relu",
            batch_first=True,
            norm_first=False
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=N
        )

        # Decoder output → vocabulary logits
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
        )

    def make_causal_mask(
        self,
        target_len,
        device
    ):
       

        return torch.triu(
            torch.ones(
                target_len,
                target_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

    def forward(
        self,
        source_input_ids,
        decoder_input_ids
    ):
      

        source_padding_mask = (
            source_input_ids == self.pad_token_id
        )

        target_padding_mask = (
            decoder_input_ids == self.pad_token_id
        )



        source = self.embedding(
            source_input_ids
        )

        source = (
            source * math.sqrt(self.d_model)
        )

        source = self.positional_encoding(
            source
        )

      

        memory = self.encoder(
            source,
            src_key_padding_mask=source_padding_mask
        )

       
     

        target = self.embedding(
            decoder_input_ids
        )

        target = (
            target * math.sqrt(self.d_model)
        )

        target = self.positional_encoding(
            target
        )

        

        target_len = decoder_input_ids.size(1)

        causal_mask = self.make_causal_mask(
            target_len=target_len,
            device=decoder_input_ids.device
        )

       

        decoder_output = self.decoder(
            tgt=target,

            memory=memory,

            tgt_mask=causal_mask,
         
            tgt_key_padding_mask=target_padding_mask,

            memory_key_padding_mask=source_padding_mask
        )

       

        logits = self.output_projection(
            decoder_output
        )

      

        return logits, memory, decoder_output, source, target

In [6]:
model = TranslationTransformer(
    vocab_size=vocab_size,
    pad_token_id=token_to_id["<PAD>"], 
    d_model=d_model,
    h=h,
    N=N,
    d_ff=d_ff,
    dropout_p=dropout_p,
    max_length=max_length
).to(device)



### Last projection linear parameter updta

### using loss.backward

In [7]:
output_linear_candidates = [
    (name, module)
    for name, module in model.named_modules()
    if (
        isinstance(module, nn.Linear)
        and module.out_features == vocab_size
    )
]

if not output_linear_candidates:
    raise RuntimeError(
        f"out_features={vocab_size}인 Linear 레이어를 찾지 못했습니다."
    )

last_linear_name, last_linear = output_linear_candidates[-1]

if last_linear.bias is None:
    raise RuntimeError("마지막 Linear에 bias가 없습니다.")

print("마지막 Linear:", last_linear_name)
print("W shape:", tuple(last_linear.weight.shape))
print("b shape:", tuple(last_linear.bias.shape))


learning_rate = 1e-3
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2),
    eps=epsilon,
    weight_decay=0.0,
)



model.train()
optimizer.zero_grad(set_to_none=True)

model_output, encoder_output, decoder_output, source, target = model(
    source_input_ids,
    decoder_input_ids,
)


print("Encoder output:", encoder_output.shape)
print("Decoder output:", decoder_output.shape)
print("Logits:", model_output.shape)
print("Labels:", labels.shape)

print("\nModel output shape:", tuple(model_output.shape))
print("Labels shape:", tuple(labels.shape))



with torch.no_grad():
    probability_sum = model_output.sum(dim=-1)

    output_is_probability = (
        model_output.min().item() >= 0.0
        and model_output.max().item() <= 1.0
        and torch.allclose(
            probability_sum,
            torch.ones_like(probability_sum),
            atol=1e-4,
            rtol=1e-4,
        )
    )

flat_output = model_output.reshape(-1, vocab_size)
flat_labels = labels.reshape(-1)

if output_is_probability:
    print("모델 출력 형식: probabilities (Softmax 적용됨)")

    loss = F.nll_loss(
        torch.log(flat_output.clamp_min(1e-12)),
        flat_labels,
        ignore_index=pad_token_id,
        reduction="mean",
        label_smoothing=0.1,

    )

else:
    print("모델 출력 형식: raw logits")

    loss = F.cross_entropy(
        flat_output,
        flat_labels,
        ignore_index=pad_token_id,
        reduction="mean",
        label_smoothing=0.1,
    )

print("Loss:", loss.item())



W_before = last_linear.weight.detach().clone()
b_before = last_linear.bias.detach().clone()


loss.backward()

if last_linear.weight.grad is None:
    raise RuntimeError("마지막 Linear의 weight gradient가 없습니다.")

if last_linear.bias.grad is None:
    raise RuntimeError("마지막 Linear의 bias gradient가 없습니다.")


# optimizer.step()에 사용될 gradient 저장
dW = last_linear.weight.grad.detach().clone()
db = last_linear.bias.grad.detach().clone()


print("\n" + "=" * 80)
print("마지막 Linear의 Gradient")
print("=" * 80)

print("\ndW shape:", tuple(dW.shape))
print("dW:")
print(dW.cpu())

print("\ndb shape:", tuple(db.shape))
print("db:")
print(db.cpu())



optimizer.step()


W_after = last_linear.weight.detach().clone()
b_after = last_linear.bias.detach().clone()

delta_W = W_after - W_before
delta_b = b_after - b_before



print("\n" + "=" * 80)
print("업데이트 전 마지막 Linear 파라미터")
print("=" * 80)

print("\nW_before:")
print(W_before.cpu())

print("\nb_before:")
print(b_before.cpu())


print("\n" + "=" * 80)
print("Adam 업데이트 변화량")
print("=" * 80)

print("\ndelta_W = W_after - W_before:")
print(delta_W.cpu())

print("\ndelta_b = b_after - b_before:")
print(delta_b.cpu())


print("\n" + "=" * 80)
print("업데이트된 마지막 Linear 파라미터")
print("=" * 80)

print("\nW_after:")
print(W_after.cpu())

print("\nb_after:")
print(b_after.cpu())



print("\n" + "=" * 80)
print("업데이트 확인")
print("=" * 80)

print(
    "W가 변경되었는가:",
    not torch.equal(W_before, W_after),
)

print(
    "b가 변경되었는가:",
    not torch.equal(b_before, b_after),
)

print(
    "W 최대 변화량:",
    delta_W.abs().max().item(),
)

print(
    "b 최대 변화량:",
    delta_b.abs().max().item(),
)

print(
    "dW norm:",
    dW.norm().item(),
)

print(
    "db norm:",
    db.norm().item(),
)



W_adam_state = optimizer.state[last_linear.weight]
b_adam_state = optimizer.state[last_linear.bias]

print("\n" + "=" * 80)
print("Adam 내부 상태")
print("=" * 80)

print("\nW Adam step:", W_adam_state["step"])
print("W 1차 모멘트 m:")
print(W_adam_state["exp_avg"].detach().cpu())

print("\nW 2차 모멘트 v:")
print(W_adam_state["exp_avg_sq"].detach().cpu())

print("\nb Adam step:", b_adam_state["step"])
print("b 1차 모멘트 m:")
print(b_adam_state["exp_avg"].detach().cpu())

print("\nb 2차 모멘트 v:")
print(b_adam_state["exp_avg_sq"].detach().cpu())

마지막 Linear: output_projection
W shape: (16, 4)
b shape: (16,)
Encoder output: torch.Size([1, 8, 4])
Decoder output: torch.Size([1, 8, 4])
Logits: torch.Size([1, 8, 16])
Labels: torch.Size([1, 8])

Model output shape: (1, 8, 16)
Labels shape: (1, 8)
모델 출력 형식: raw logits
Loss: 2.895735502243042

마지막 Linear의 Gradient

dW shape: (16, 4)
dW:
tensor([[-2.1690e-03,  5.6029e-05, -5.9083e-03,  8.0213e-03],
        [ 1.8338e-02, -5.7921e-02, -7.9835e-02,  1.1942e-01],
        [ 1.4933e-01, -1.9091e-02, -1.2624e-01, -3.9935e-03],
        [-4.6425e-03, -7.1043e-03, -2.6969e-02,  3.8716e-02],
        [-1.9134e-02,  2.4634e-02, -2.9641e-02,  2.4142e-02],
        [-7.0763e-03, -3.1840e-02, -5.6143e-02,  9.5059e-02],
        [-1.7147e-02, -1.5022e-02, -1.6537e-02,  4.8707e-02],
        [-2.3123e-02,  5.8829e-03, -1.3744e-02,  3.0984e-02],
        [-2.2450e-02, -5.7146e-03, -7.2658e-02,  1.0082e-01],
        [ 5.5328e-02, -6.6920e-02,  7.8719e-02, -6.7127e-02],
        [ 1.2636e-01,  7.3807e-02, -8.225

### using formulation

In [8]:
Z = model_output
Memory = encoder_output
H = decoder_output


print("source(encoder_input): \n")
print(source,'\n')

print("targeet(decoder_input): \n")
print(target,'\n')

print("Z\n")
print(Z,'\n')
print("Memory\n")
print(Memory,'\n')
print("H\n")
print(H,'\n')




source(encoder_input): 

tensor([[[ 0.6487, -1.2163, -0.7753, -1.3020],
         [-1.0037, -0.1971,  1.6114, -1.4977],
         [ 0.7112, -1.9374, -1.0376,  3.1914],
         [ 0.1190, -4.0166, -4.2832, -1.3778],
         [-0.0000, -1.8619, -2.6223, -3.7018],
         [ 1.1327,  1.4449, -0.9291, -0.0000],
         [-3.8159, -0.1841,  1.1292,  4.5258],
         [-0.7141,  4.0311, -2.4786,  3.1857]]], device='mps:0',
       grad_fn=<MulBackward0>) 

targeet(decoder_input): 

tensor([[[-2.7792, -2.1784,  7.5450,  3.5618],
         [ 0.7345, -0.9422, -1.4445,  1.8767],
         [-0.8474,  0.0000, -4.5417,  2.8231],
         [ 3.4205, -1.2939, -0.0000,  1.4415],
         [-0.0000,  3.4065, -1.3507, -2.7327],
         [ 2.7336, -3.4577,  1.4787, -2.0072],
         [ 1.2236,  2.5053,  2.0086,  2.7173],
         [-2.7755, -0.4133,  1.1403,  4.5251]]], device='mps:0',
       grad_fn=<MulBackward0>) 

Z

tensor([[[-1.2191,  0.4504,  0.3494, -0.2365, -0.9209,  0.7237,  0.6723,
           0.1702, 

In [9]:
criterion = nn.CrossEntropyLoss(
    ignore_index = token_to_id["<PAD>"],
    label_smoothing = 0.1
)

loss = criterion(
    model_output.reshape(-1, model_output.size(-1)),
    labels.reshape(-1)
)

print("loss:", loss.item())

loss: 2.895735502243042


In [10]:
P = torch.softmax(Z, dim=-1)
P




tensor([[[0.0142, 0.0753, 0.0680, 0.0379, 0.0191, 0.0989, 0.0940, 0.0569,
          0.0899, 0.0698, 0.0379, 0.0672, 0.0862, 0.0380, 0.0108, 0.1359],
         [0.0108, 0.1452, 0.1072, 0.0467, 0.0168, 0.1064, 0.0443, 0.0219,
          0.1176, 0.1090, 0.0181, 0.0863, 0.0374, 0.0197, 0.0223, 0.0904],
         [0.0150, 0.0666, 0.1549, 0.0386, 0.0615, 0.0598, 0.0242, 0.0335,
          0.1014, 0.0793, 0.0427, 0.0587, 0.0741, 0.0468, 0.0482, 0.0945],
         [0.0126, 0.2035, 0.0646, 0.0520, 0.0087, 0.1347, 0.0579, 0.0204,
          0.1122, 0.1083, 0.0127, 0.0811, 0.0279, 0.0161, 0.0187, 0.0686],
         [0.0196, 0.0447, 0.1389, 0.0341, 0.0949, 0.0443, 0.0186, 0.0422,
          0.0843, 0.0600, 0.0601, 0.0416, 0.0970, 0.0719, 0.0668, 0.0809],
         [0.0209, 0.2453, 0.0392, 0.0599, 0.0069, 0.1488, 0.0622, 0.0233,
          0.1032, 0.0973, 0.0120, 0.0629, 0.0262, 0.0199, 0.0255, 0.0465],
         [0.0109, 0.0788, 0.1492, 0.0383, 0.0379, 0.0740, 0.0355, 0.0307,
          0.1074, 0.0891, 0.0334

In [11]:


V = 16
epsilon = 0.1

# labels shape: [1, 8]
one_hot_labels = F.one_hot(
    labels,
    num_classes=V,
).to(dtype=model_output.dtype)

Q = (
    (1.0 - epsilon) * one_hot_labels
    + epsilon / V
)

In [12]:
G = (P[0] - Q[0])/target.shape[1]
G

tensor([[ 9.9096e-04,  8.6285e-03,  7.7243e-03,  3.9531e-03,  1.6068e-03,
          1.1586e-02,  1.0966e-02,  6.3289e-03,  1.0458e-02,  7.9411e-03,
         -1.0854e-01,  7.6180e-03,  9.9985e-03,  3.9660e-03,  5.7248e-04,
          1.6203e-02],
        [ 5.6546e-04,  1.7373e-02,  1.2613e-02,  5.0528e-03,  1.3226e-03,
          1.2519e-02,  4.7527e-03,  1.9545e-03,  1.3916e-02,  1.2844e-02,
          1.4849e-03, -1.0249e-01,  3.8932e-03,  1.6804e-03,  2.0005e-03,
          1.0518e-02],
        [ 1.0969e-03,  7.5457e-03,  1.8583e-02,  4.0441e-03,  6.9104e-03,
          6.6890e-03,  2.2389e-03,  3.4120e-03,  1.1890e-02,  9.1344e-03,
          4.5609e-03,  6.5596e-03, -1.0401e-01,  5.0725e-03,  5.2492e-03,
          1.1027e-02],
        [ 7.9701e-04,  2.4658e-02,  7.2933e-03,  5.7238e-03,  3.0468e-04,
          1.6056e-02,  6.4570e-03,  1.7665e-03,  1.3238e-02,  1.2756e-02,
          8.0206e-04,  9.3572e-03,  2.7037e-03, -1.1127e-01,  1.5577e-03,
          7.7995e-03],
        [ 1.6660e-03

In [13]:
dW_hand = G.T @ H[0]

dW_hand

tensor([[-2.1690e-03,  5.6028e-05, -5.9083e-03,  8.0213e-03],
        [ 1.8338e-02, -5.7921e-02, -7.9835e-02,  1.1942e-01],
        [ 1.4933e-01, -1.9091e-02, -1.2624e-01, -3.9935e-03],
        [-4.6425e-03, -7.1043e-03, -2.6969e-02,  3.8716e-02],
        [-1.9134e-02,  2.4634e-02, -2.9641e-02,  2.4142e-02],
        [-7.0763e-03, -3.1840e-02, -5.6143e-02,  9.5059e-02],
        [-1.7147e-02, -1.5022e-02, -1.6537e-02,  4.8707e-02],
        [-2.3123e-02,  5.8829e-03, -1.3744e-02,  3.0984e-02],
        [-2.2450e-02, -5.7146e-03, -7.2658e-02,  1.0082e-01],
        [ 5.5328e-02, -6.6920e-02,  7.8719e-02, -6.7127e-02],
        [ 1.2636e-01,  7.3807e-02, -8.2258e-02, -1.1790e-01],
        [-2.5743e-02,  2.7974e-02,  9.9506e-02, -1.0174e-01],
        [-1.3131e-02, -8.6432e-02,  1.3498e-01, -3.5414e-02],
        [-7.9435e-02,  1.3204e-01,  7.4405e-02, -1.2701e-01],
        [ 3.8363e-02, -1.4477e-01,  1.2569e-01, -1.9283e-02],
        [-1.7366e-01,  1.7042e-01, -3.3584e-03,  6.5968e-03]], device=

In [14]:
dW

tensor([[-2.1690e-03,  5.6029e-05, -5.9083e-03,  8.0213e-03],
        [ 1.8338e-02, -5.7921e-02, -7.9835e-02,  1.1942e-01],
        [ 1.4933e-01, -1.9091e-02, -1.2624e-01, -3.9935e-03],
        [-4.6425e-03, -7.1043e-03, -2.6969e-02,  3.8716e-02],
        [-1.9134e-02,  2.4634e-02, -2.9641e-02,  2.4142e-02],
        [-7.0763e-03, -3.1840e-02, -5.6143e-02,  9.5059e-02],
        [-1.7147e-02, -1.5022e-02, -1.6537e-02,  4.8707e-02],
        [-2.3123e-02,  5.8829e-03, -1.3744e-02,  3.0984e-02],
        [-2.2450e-02, -5.7146e-03, -7.2658e-02,  1.0082e-01],
        [ 5.5328e-02, -6.6920e-02,  7.8719e-02, -6.7127e-02],
        [ 1.2636e-01,  7.3807e-02, -8.2258e-02, -1.1790e-01],
        [-2.5743e-02,  2.7974e-02,  9.9506e-02, -1.0174e-01],
        [-1.3131e-02, -8.6432e-02,  1.3498e-01, -3.5414e-02],
        [-7.9435e-02,  1.3204e-01,  7.4405e-02, -1.2701e-01],
        [ 3.8363e-02, -1.4477e-01,  1.2569e-01, -1.9283e-02],
        [-1.7366e-01,  1.7042e-01, -3.3584e-03,  6.5968e-03]], device=

In [15]:
db_hand = G.sum(dim=0)

db_hand


tensor([ 0.0087,  0.1060, -0.0173,  0.0359,  0.0304,  0.0846,  0.0427,  0.0311,
         0.0927, -0.0356, -0.0834, -0.0535, -0.0506, -0.0802, -0.0894, -0.0222],
       device='mps:0', grad_fn=<SumBackward1>)

In [16]:
db

tensor([ 0.0087,  0.1060, -0.0173,  0.0359,  0.0304,  0.0846,  0.0427,  0.0311,
         0.0927, -0.0356, -0.0834, -0.0535, -0.0506, -0.0802, -0.0894, -0.0222],
       device='mps:0')

In [17]:
dW_hand = torch.as_tensor(
    dW_hand,
    device=last_linear.weight.device,
    dtype=last_linear.weight.dtype,
).detach().clone()

db_hand = torch.as_tensor(
    db_hand,
    device=last_linear.bias.device,
    dtype=last_linear.bias.dtype,
).detach().clone()


# PyTorch Linear weight는 [out_features, in_features]
if dW_hand.shape != last_linear.weight.shape:
    if (
        dW_hand.ndim == 2
        and dW_hand.T.shape == last_linear.weight.shape
    ):
        print("dW_hand를 PyTorch weight 방향에 맞게 전치합니다.")
        dW_hand = dW_hand.T.contiguous()
    else:
        raise RuntimeError(
            f"dW_hand shape {tuple(dW_hand.shape)}와 "
            f"weight shape {tuple(last_linear.weight.shape)}가 다릅니다."
        )

if db_hand.numel() != last_linear.bias.numel():
    raise RuntimeError(
        f"db_hand shape {tuple(db_hand.shape)}와 "
        f"bias shape {tuple(last_linear.bias.shape)}가 다릅니다."
    )

db_hand = db_hand.reshape_as(last_linear.bias)

assert torch.isfinite(dW_hand).all()
assert torch.isfinite(db_hand).all()



learning_rate = 1e-3
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
step_number = 1



W_before_hand = W_before
b_before_hand = b_before

mW_previous_hand = torch.zeros_like(W_before_hand)
vW_previous_hand = torch.zeros_like(W_before_hand)

mb_previous_hand = torch.zeros_like(b_before_hand)
vb_previous_hand = torch.zeros_like(b_before_hand)



mW_hand = (
    beta1 * mW_previous_hand
    + (1.0 - beta1) * dW_hand
)

mb_hand = (
    beta1 * mb_previous_hand
    + (1.0 - beta1) * db_hand
)



vW_hand = (
    beta2 * vW_previous_hand
    + (1.0 - beta2) * dW_hand.square()
)

vb_hand = (
    beta2 * vb_previous_hand
    + (1.0 - beta2) * db_hand.square()
)


mW_hat_hand = mW_hand / (
    1.0 - beta1**step_number
)

vW_hat_hand = vW_hand / (
    1.0 - beta2**step_number
)

mb_hat_hand = mb_hand / (
    1.0 - beta1**step_number
)

vb_hat_hand = vb_hand / (
    1.0 - beta2**step_number
)



W_after_hand = (
    W_before_hand
    - learning_rate
    * mW_hat_hand
    / (torch.sqrt(vW_hat_hand) + epsilon)
)

b_after_hand = (
    b_before_hand
    - learning_rate
    * mb_hat_hand
    / (torch.sqrt(vb_hat_hand) + epsilon)
)




delta_W_hand = W_after_hand - W_before_hand
delta_b_hand = b_after_hand - b_before_hand



print("\n" + "=" * 80)
print("수동 계산 Gradient")
print("=" * 80)

print("\ndW_hand:")
print(dW_hand.cpu())

print("\ndb_hand:")
print(db_hand.cpu())

print("\ndW_hand shape:", tuple(dW_hand.shape))
print("db_hand shape:", tuple(db_hand.shape))


print("\n" + "=" * 80)
print("업데이트 전 파라미터")
print("=" * 80)

print("\nW_before_hand:")
print(W_before_hand.cpu())

print("\nb_before_hand:")
print(b_before_hand.cpu())


print("\n" + "=" * 80)
print("Adam 모멘트")
print("=" * 80)

print("\nmW_hand:")
print(mW_hand.cpu())

print("\nvW_hand:")
print(vW_hand.cpu())

print("\nmb_hand:")
print(mb_hand.cpu())

print("\nvb_hand:")
print(vb_hand.cpu())


print("\n" + "=" * 80)
print("파라미터 변화량")
print("=" * 80)

print("\ndelta_W_hand:")
print(delta_W_hand.cpu())

print("\ndelta_b_hand:")
print(delta_b_hand.cpu())


print("\n" + "=" * 80)
print("수동 Adam 업데이트 후 파라미터")
print("=" * 80)

print("\nW_after_hand:")
print(W_after_hand.cpu())

print("\nb_after_hand:")
print(b_after_hand.cpu())






수동 계산 Gradient

dW_hand:
tensor([[-2.1690e-03,  5.6028e-05, -5.9083e-03,  8.0213e-03],
        [ 1.8338e-02, -5.7921e-02, -7.9835e-02,  1.1942e-01],
        [ 1.4933e-01, -1.9091e-02, -1.2624e-01, -3.9935e-03],
        [-4.6425e-03, -7.1043e-03, -2.6969e-02,  3.8716e-02],
        [-1.9134e-02,  2.4634e-02, -2.9641e-02,  2.4142e-02],
        [-7.0763e-03, -3.1840e-02, -5.6143e-02,  9.5059e-02],
        [-1.7147e-02, -1.5022e-02, -1.6537e-02,  4.8707e-02],
        [-2.3123e-02,  5.8829e-03, -1.3744e-02,  3.0984e-02],
        [-2.2450e-02, -5.7146e-03, -7.2658e-02,  1.0082e-01],
        [ 5.5328e-02, -6.6920e-02,  7.8719e-02, -6.7127e-02],
        [ 1.2636e-01,  7.3807e-02, -8.2258e-02, -1.1790e-01],
        [-2.5743e-02,  2.7974e-02,  9.9506e-02, -1.0174e-01],
        [-1.3131e-02, -8.6432e-02,  1.3498e-01, -3.5414e-02],
        [-7.9435e-02,  1.3204e-01,  7.4405e-02, -1.2701e-01],
        [ 3.8363e-02, -1.4477e-01,  1.2569e-01, -1.9283e-02],
        [-1.7366e-01,  1.7042e-01, -3.3584e-

In [18]:
dW

tensor([[-2.1690e-03,  5.6029e-05, -5.9083e-03,  8.0213e-03],
        [ 1.8338e-02, -5.7921e-02, -7.9835e-02,  1.1942e-01],
        [ 1.4933e-01, -1.9091e-02, -1.2624e-01, -3.9935e-03],
        [-4.6425e-03, -7.1043e-03, -2.6969e-02,  3.8716e-02],
        [-1.9134e-02,  2.4634e-02, -2.9641e-02,  2.4142e-02],
        [-7.0763e-03, -3.1840e-02, -5.6143e-02,  9.5059e-02],
        [-1.7147e-02, -1.5022e-02, -1.6537e-02,  4.8707e-02],
        [-2.3123e-02,  5.8829e-03, -1.3744e-02,  3.0984e-02],
        [-2.2450e-02, -5.7146e-03, -7.2658e-02,  1.0082e-01],
        [ 5.5328e-02, -6.6920e-02,  7.8719e-02, -6.7127e-02],
        [ 1.2636e-01,  7.3807e-02, -8.2258e-02, -1.1790e-01],
        [-2.5743e-02,  2.7974e-02,  9.9506e-02, -1.0174e-01],
        [-1.3131e-02, -8.6432e-02,  1.3498e-01, -3.5414e-02],
        [-7.9435e-02,  1.3204e-01,  7.4405e-02, -1.2701e-01],
        [ 3.8363e-02, -1.4477e-01,  1.2569e-01, -1.9283e-02],
        [-1.7366e-01,  1.7042e-01, -3.3584e-03,  6.5968e-03]], device=

In [19]:
dW_hand

tensor([[-2.1690e-03,  5.6028e-05, -5.9083e-03,  8.0213e-03],
        [ 1.8338e-02, -5.7921e-02, -7.9835e-02,  1.1942e-01],
        [ 1.4933e-01, -1.9091e-02, -1.2624e-01, -3.9935e-03],
        [-4.6425e-03, -7.1043e-03, -2.6969e-02,  3.8716e-02],
        [-1.9134e-02,  2.4634e-02, -2.9641e-02,  2.4142e-02],
        [-7.0763e-03, -3.1840e-02, -5.6143e-02,  9.5059e-02],
        [-1.7147e-02, -1.5022e-02, -1.6537e-02,  4.8707e-02],
        [-2.3123e-02,  5.8829e-03, -1.3744e-02,  3.0984e-02],
        [-2.2450e-02, -5.7146e-03, -7.2658e-02,  1.0082e-01],
        [ 5.5328e-02, -6.6920e-02,  7.8719e-02, -6.7127e-02],
        [ 1.2636e-01,  7.3807e-02, -8.2258e-02, -1.1790e-01],
        [-2.5743e-02,  2.7974e-02,  9.9506e-02, -1.0174e-01],
        [-1.3131e-02, -8.6432e-02,  1.3498e-01, -3.5414e-02],
        [-7.9435e-02,  1.3204e-01,  7.4405e-02, -1.2701e-01],
        [ 3.8363e-02, -1.4477e-01,  1.2569e-01, -1.9283e-02],
        [-1.7366e-01,  1.7042e-01, -3.3584e-03,  6.5968e-03]], device=

In [20]:
db

tensor([ 0.0087,  0.1060, -0.0173,  0.0359,  0.0304,  0.0846,  0.0427,  0.0311,
         0.0927, -0.0356, -0.0834, -0.0535, -0.0506, -0.0802, -0.0894, -0.0222],
       device='mps:0')

In [21]:
db_hand

tensor([ 0.0087,  0.1060, -0.0173,  0.0359,  0.0304,  0.0846,  0.0427,  0.0311,
         0.0927, -0.0356, -0.0834, -0.0535, -0.0506, -0.0802, -0.0894, -0.0222],
       device='mps:0')

In [22]:
W_after

tensor([[ 0.2946,  0.0716,  0.2753, -0.3972],
        [ 0.4173, -0.4433, -0.1045,  0.3440],
        [-0.0595,  0.3388, -0.2590,  0.4658],
        [ 0.4847,  0.1161,  0.2552,  0.3734],
        [-0.4278,  0.4501, -0.3275, -0.4803],
        [ 0.2963, -0.2379,  0.1496,  0.4227],
        [-0.1073, -0.4370,  0.2474,  0.2884],
        [-0.0861,  0.1956,  0.3982, -0.1111],
        [ 0.0282, -0.1577, -0.1793,  0.2027],
        [ 0.1042, -0.2053, -0.2079,  0.3321],
        [-0.4391,  0.1919, -0.0256, -0.4127],
        [-0.0597, -0.2524, -0.2209,  0.4516],
        [-0.0897,  0.4683,  0.3764,  0.0361],
        [ 0.0106,  0.4207,  0.3045, -0.3656],
        [ 0.4792,  0.4303, -0.0772, -0.3625],
        [-0.2851,  0.0399, -0.0055,  0.4288]], device='mps:0')

In [23]:
W_after_hand

tensor([[ 0.2946,  0.0716,  0.2753, -0.3972],
        [ 0.4173, -0.4433, -0.1045,  0.3440],
        [-0.0595,  0.3388, -0.2590,  0.4658],
        [ 0.4847,  0.1161,  0.2552,  0.3734],
        [-0.4278,  0.4501, -0.3275, -0.4803],
        [ 0.2963, -0.2379,  0.1496,  0.4227],
        [-0.1073, -0.4370,  0.2474,  0.2884],
        [-0.0861,  0.1956,  0.3982, -0.1111],
        [ 0.0282, -0.1577, -0.1793,  0.2027],
        [ 0.1042, -0.2053, -0.2079,  0.3321],
        [-0.4391,  0.1919, -0.0256, -0.4127],
        [-0.0597, -0.2524, -0.2209,  0.4516],
        [-0.0897,  0.4683,  0.3764,  0.0361],
        [ 0.0106,  0.4207,  0.3045, -0.3656],
        [ 0.4792,  0.4303, -0.0772, -0.3625],
        [-0.2851,  0.0399, -0.0055,  0.4288]], device='mps:0')

In [24]:
b_after

tensor([-0.4406,  0.3857,  0.0024, -0.1614, -0.4496,  0.3592, -0.2146,  0.0733,
         0.4203,  0.0985, -0.1665, -0.3287,  0.4545,  0.2993, -0.1156,  0.1370],
       device='mps:0')

In [25]:
b_after_hand

tensor([-0.4406,  0.3857,  0.0024, -0.1614, -0.4496,  0.3592, -0.2146,  0.0733,
         0.4203,  0.0985, -0.1665, -0.3287,  0.4545,  0.2993, -0.1156,  0.1370],
       device='mps:0')

### Decoder norm3 update (\gamma,\beta)

In [ ]:
model = TranslationTransformer(
    vocab_size=vocab_size,
    pad_token_id=token_to_id["<PAD>"], 
    d_model=d_model,
    h=h,
    N=N,
    d_ff=d_ff,
    dropout_p=dropout_p,
    max_length=max_length
).to(device)



In [30]:
norm3 = model.decoder.layers[0].norm3

gamma =  norm3.weight
beta = norm3.bias

print(gamma, gamma.shape)
print()
print(beta, beta.shape)


Parameter containing:
tensor([0.9990, 0.9990, 0.9990, 0.9990], device='mps:0', requires_grad=True) torch.Size([4])

Parameter containing:
tensor([-0.0010,  0.0010,  0.0010, -0.0010], device='mps:0',
       requires_grad=True) torch.Size([4])
